In [1]:
from torch.utils.data import DataLoader
import torch
from sklearn.utils.class_weight import compute_class_weight
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup,AutoTokenizer,DistilBertModel
from torch.nn.modules.loss import BCEWithLogitsLoss,CrossEntropyLoss
import numpy as np
from torchvision.models import resnet18,ResNet18_Weights
from torchvision import transforms
from modules import CollateFunction,creation_dataframe,CreationDataset,Train,DistilbertResnetModel

In [2]:
#Creation of the dataframes from the jsonl files
train_df=creation_dataframe("../data/train.jsonl")
val_df=creation_dataframe("../data/dev.jsonl")

In [3]:
#Creation of the datasets
train_dataset=CreationDataset(train_df,"../CLIP_model/modules/clip_embeddings/train_clip_embeddings.pt")
val_dataset=CreationDataset(val_df,"../CLIP_model/modules/clip_embeddings/val_clip_embeddings.pt")

In [4]:
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [5]:
collate_object=CollateFunction(tokenizer)

In [6]:
#Creation of the dataloaders
batch_size=32
train_dataloader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True,collate_fn=collate_object.collate_fn,drop_last=True)
val_dataloader=DataLoader(val_dataset,batch_size=batch_size,shuffle=True,collate_fn=collate_object.collate_fn,drop_last=True)

In [7]:
#Use of the resnet18 model initialized with its default pretrained weights as the Vision model
resnet_model=resnet18(weights=ResNet18_Weights.DEFAULT)

In [8]:
#Use of the pretrained distilbert model as the transformer model
distilbert_model=DistilBertModel.from_pretrained("distilbert-base-uncased")

In [9]:
#Use of the class weights to compensate imabalances of the dataset and make more accurate predictions
class_weight=compute_class_weight("balanced",classes=np.unique(train_df["label"]),y=train_df["label"].to_numpy())
class_weight=torch.tensor(class_weight,dtype=torch.float32)
print(class_weight)

tensor([0.7798, 1.3934])


In [10]:
#Finally, use of the custom model
#Training hyperparameters
model=DistilbertResnetModel(distilbert_model,resnet_model,with_clip_image=True,with_clip_text=True)
n_epochs=10
n_steps=len(train_dataloader)*n_epochs
n_warmup_steps=int(0.1*n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
trainer=Train(model=model,loss_fn=loss_fn,n_epochs=n_epochs,device=device,n_steps=n_steps,n_warmup_steps=n_warmup_steps,n_frozen_distilbert_layers=6,n_frozen_resnet_layers=4)
trainer.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader,path="./modules/train_savings/with_clip",with_clip_images=True,with_clip_text=True)

2026-03-12 19:45:22.461 | INFO     | modules.train:run_training:167 - Epoch 0 :
2026-03-12 20:07:15.273 | INFO     | modules.train:run_training:257 - Epoch 0: Train Loss = 0.7171681595298479
2026-03-12 20:07:15.273 | INFO     | modules.train:run_training:258 - Epoch 0: Train Accuracy = 0.48785377358490567
2026-03-12 20:07:15.273 | INFO     | modules.train:run_training:259 - Epoch 0: Train F1 = 0.49432437007282815
2026-03-12 20:07:15.273 | INFO     | modules.train:run_training:261 - Epoch 0: Validation Loss = 0.6976184209187826
2026-03-12 20:07:15.281 | INFO     | modules.train:run_training:262 - Epoch 0: Validation Accuracy = 0.48125
2026-03-12 20:07:15.281 | INFO     | modules.train:run_training:263 - Epoch 0: Validation F1 = 0.47301376616272345
2026-03-12 20:07:16.038 | INFO     | modules.train:run_training:167 - Epoch 1 :
2026-03-12 20:33:16.989 | INFO     | modules.train:run_training:257 - Epoch 1: Train Loss = 0.6410010304091112
2026-03-12 20:33:16.989 | INFO     | modules.train:r

In [12]:
#Finally, use of the custom model
#Training hyperparameters
model=DistilbertResnetModel(distilbert_model,resnet_model,with_clip_image=False,with_clip_text=False)
n_epochs=10
n_steps=len(train_dataloader)*n_epochs
n_warmup_steps=int(0.1*n_steps)
loss_fn=CrossEntropyLoss(weight=class_weight)
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
trainer=Train(model=model,loss_fn=loss_fn,n_epochs=n_epochs,device=device,n_steps=n_steps,n_warmup_steps=n_warmup_steps,n_frozen_distilbert_layers=6,n_frozen_resnet_layers=4)
trainer.run_training(train_dataloader=train_dataloader,val_dataloader=val_dataloader,path="./modules/train_savings/without_clip",with_clip_images=False,with_clip_text=False)

2026-03-12 21:35:59.104 | INFO     | modules.train:run_training:167 - Epoch 0 :


2026-03-12 22:30:47.765 | INFO     | modules.train:run_training:257 - Epoch 0: Train Loss = 0.7226507054184967
2026-03-12 22:30:47.778 | INFO     | modules.train:run_training:258 - Epoch 0: Train Accuracy = 0.4858490566037736
2026-03-12 22:30:47.781 | INFO     | modules.train:run_training:259 - Epoch 0: Train F1 = 0.4950835180842529
2026-03-12 22:30:47.782 | INFO     | modules.train:run_training:261 - Epoch 0: Validation Loss = 0.7056100606918335
2026-03-12 22:30:47.783 | INFO     | modules.train:run_training:262 - Epoch 0: Validation Accuracy = 0.49166666666666664
2026-03-12 22:30:47.783 | INFO     | modules.train:run_training:263 - Epoch 0: Validation F1 = 0.49117190396737115
2026-03-12 22:30:49.556 | INFO     | modules.train:run_training:167 - Epoch 1 :
2026-03-12 23:03:30.247 | INFO     | modules.train:run_training:257 - Epoch 1: Train Loss = 0.6447255177317925
2026-03-12 23:03:30.250 | INFO     | modules.train:run_training:258 - Epoch 1: Train Accuracy = 0.6253537735849056
2026-03